# Azure OpenAI + Tools: Case Assistant Notebook

This notebook matches your architecture:
- One Azure OpenAI entry point with tool-style routing.
- `get_case_list` tool builds only the **OData `$filter` expression** from natural language.
- `get_case_details` and `follow_case` are separate tools.

The key safety requirement is enforced: model output for filter generation is **only the filter expression text** (no prose, no `$filter=` prefix).


In [ ]:
import json
import os
from textwrap import dedent
from urllib.parse import quote

import requests
from openai import AzureOpenAI


In [ ]:
# Azure OpenAI configuration
# Required environment variables:
# - AZURE_OPENAI_API_KEY
# - AZURE_OPENAI_ENDPOINT          e.g. https://<resource>.openai.azure.com
# - AZURE_OPENAI_DEPLOYMENT        e.g. gpt-4.1-mini
# Optional:
# - AZURE_OPENAI_API_VERSION       default: 2024-10-21

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1-mini")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")

assert AZURE_OPENAI_API_KEY, "Missing AZURE_OPENAI_API_KEY"
assert AZURE_OPENAI_ENDPOINT, "Missing AZURE_OPENAI_ENDPOINT"

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)


In [ ]:
# OData + API settings (edit for your environment)
CASE_ODATA_BASE_URL = "https://example.contoso.com/odata/Cases"
CASE_DETAILS_API_URL = "https://example.contoso.com/api/cases/{case_number}"
FOLLOW_CASE_API_URL = "https://example.contoso.com/api/cases/{case_number}/followers"

DEFAULT_SELECT_FIELDS = [
    "CaseNumber",
    "Title",
    "CustomerName",
    "Status",
    "Severity",
    "CreatedDate",
    "AssignedTo",
]


In [ ]:
def build_filter_system_prompt(field_details: str, instruction_details: str, examples: list[dict] | None = None) -> str:
    examples = examples or []
    ex = []
    for i, item in enumerate(examples, start=1):
        ex.append(f"Example {i}\nQuestion: {item['question']}\nFilter: {item['filter']}")

    return dedent(f"""
    You convert user questions into ONLY an OData $filter expression.

    HARD RULES:
    - Return only a valid OData filter expression string.
    - Do NOT return '$filter='.
    - Do NOT include markdown, explanations, labels, XML/JSON, or code fences.
    - Use only known fields and mappings from metadata below.
    - Prefer explicit parentheses when combining AND/OR.
    - String literals must use single quotes.

    FIELD DETAILS:
    {field_details}

    INSTRUCTIONS:
    {instruction_details}

    FEW-SHOT EXAMPLES:
    {chr(10).join(ex) if ex else 'None'}
    """).strip()


def generate_case_filter(user_question: str, field_details: str, instruction_details: str, examples: list[dict] | None = None) -> str:
    system_prompt = build_filter_system_prompt(field_details, instruction_details, examples)

    resp = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
        temperature=0,
    )

    raw = resp.output_text.strip()

    # Defensive cleanup for safe URL assembly.
    raw = raw.replace("$filter=", "", 1).replace("Filter:", "", 1).strip().strip("`").strip()

    if "\n" in raw or not raw:
        raise ValueError(f"Expected one-line OData filter expression. Got: {raw!r}")

    forbidden = ["SELECT ", "http://", "https://", "{" , "}"]
    if any(token in raw for token in forbidden):
        raise ValueError(f"Output looks unsafe/non-filter-like: {raw!r}")

    return raw



In [ ]:
# Scenario-specific metadata for your Case domain
case_field_details = dedent("""
Entity: Cases
- CaseNumber (Edm.String): unique case id, e.g. 'CS1234567'
- CustomerName (Edm.String): customer/account name
- Title (Edm.String): case subject/title
- Status (Edm.String): New, Active, PendingCustomer, Resolved, Closed
- Severity (Edm.String): Critical, High, Medium, Low
- Priority (Edm.String): P1, P2, P3, P4
- CreatedDate (Edm.DateTimeOffset)
- AssignedTo (Edm.String): owner alias
- Product (Edm.String)
""").strip()

case_instruction_details = dedent("""
- 'open cases' means Status in ('New','Active','PendingCustomer').
- 'closed cases' means Status in ('Resolved','Closed').
- If user says 'for customer X', map to CustomerName eq 'X'.
- If user gives a case number, map to CaseNumber eq '<value>'.
- If user says 'last N days', use CreatedDate ge <utc_now_minus_N_days_iso>.
- Keep output limited to filter expression only.
""").strip()

case_examples = [
    {
        "question": "Cases for customer Dell",
        "filter": "CustomerName eq 'Dell'",
    },
    {
        "question": "open critical cases for customer Dell",
        "filter": "(Status eq 'New' or Status eq 'Active' or Status eq 'PendingCustomer') and Severity eq 'Critical' and CustomerName eq 'Dell'",
    },
]


In [ ]:
def build_case_list_url(filter_expr: str, select_fields: list[str] | None = None, top: int = 50) -> str:
    select_fields = select_fields or DEFAULT_SELECT_FIELDS
    select_clause = ",".join(select_fields)
    # URL-encode filter for safe transport.
    encoded_filter = quote(filter_expr, safe="()',$ =")
    return f"{CASE_ODATA_BASE_URL}?$select={select_clause}&$top={top}&$filter={encoded_filter}"


def get_case_list(selection_criteria: str) -> dict:
    filter_expr = generate_case_filter(
        user_question=selection_criteria,
        field_details=case_field_details,
        instruction_details=case_instruction_details,
        examples=case_examples,
    )

    url = build_case_list_url(filter_expr)

    # Uncomment this in your environment when API is reachable:
    # response = requests.get(url, timeout=30)
    # response.raise_for_status()
    # rows = response.json().get("value", [])

    rows = []  # placeholder for notebook experimentation
    return {
        "tool": "get_case_list",
        "selection_criteria": selection_criteria,
        "odata_filter": filter_expr,
        "odata_url": url,
        "case_list": rows,
    }


def get_case_details(case_number: str) -> dict:
    url = CASE_DETAILS_API_URL.format(case_number=case_number)
    # response = requests.get(url, timeout=30)
    # response.raise_for_status()
    # details = response.json()

    details = {"CaseNumber": case_number, "_note": "placeholder"}
    return {"tool": "get_case_details", "case_details": details, "url": url}


def follow_case(case_number: str, user_alias: str) -> dict:
    url = FOLLOW_CASE_API_URL.format(case_number=case_number)
    payload = {"userAlias": user_alias}

    # response = requests.post(url, json=payload, timeout=30)
    # response.raise_for_status()

    return {
        "tool": "follow_case",
        "status": "success-placeholder",
        "url": url,
        "payload": payload,
    }


In [ ]:
# Simple router to mirror "Azure OpenAI call with tools" orchestration.
# In production, you can replace this with native tool-calling and function execution.

def route_to_tool(intent: str, **kwargs):
    if intent == "get_case_list":
        return get_case_list(kwargs["selection_criteria"])
    if intent == "get_case_details":
        return get_case_details(kwargs["case_number"])
    if intent == "follow_case":
        return follow_case(kwargs["case_number"], kwargs["user_alias"])
    raise ValueError(f"Unknown intent: {intent}")


In [ ]:
# --- Experiment 1: get case list from natural-language criteria ---
result_case_list = route_to_tool(
    "get_case_list",
    selection_criteria="Show open critical cases for customer Dell",
)
print(json.dumps(result_case_list, indent=2))


In [ ]:
# --- Experiment 2: get case details ---
result_details = route_to_tool("get_case_details", case_number="CS1234567")
print(json.dumps(result_details, indent=2))


In [ ]:
# --- Experiment 3: follow a case ---
result_follow = route_to_tool(
    "follow_case",
    case_number="CS1234567",
    user_alias="jdoe",
)
print(json.dumps(result_follow, indent=2))
